# CoRVUS Karyon Tokenizer — Verification Notebook

This notebook verifies:
1. Porto Seguro data loading and preprocessing
2. Individual tokenizer behaviour
3. Training and evaluation comparison (linear / center / ordered)
4. Visualisation of learned thresholds and assignments (ordered mode)
5. Result summary

> **Setup**: ensure `requirements.txt` dependencies are installed and
> Porto Seguro `train.csv` is at `../data/porto-seguro-safe-driver-prediction/train.csv`.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import optim
from torch.utils.data import DataLoader

# Make src importable from repo root
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings('ignore')
print(f'PyTorch {torch.__version__}  |  device: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 1. Data Loading and Preprocessing

In [ ]:
from src.data.porto import load_porto_data, infer_schema

DATA_DIR = '../data/porto-seguro-safe-driver-prediction'

try:
    prep, train_ds, valid_ds = load_porto_data(DATA_DIR, valid_size=0.2, random_state=42)
    print(f'Train size : {len(train_ds)}')
    print(f'Valid size : {len(valid_ds)}')
    print(f'Numeric    : {prep.num_numeric}  -> {prep.numeric_cols[:5]} ...')
    print(f'Categorical: {prep.num_cat}      -> {prep.cat_cols[:5]} ...')
    print(f'Binary     : {prep.num_bin}      -> {prep.bin_cols[:5]} ...')
    DATA_AVAILABLE = True
except FileNotFoundError as e:
    print(f'Data not found: {e}')
    print('Falling back to synthetic data for demonstration.')
    DATA_AVAILABLE = False

In [ ]:
# If real data is not available, build a tiny synthetic dataset
if not DATA_AVAILABLE:
    import pandas as pd
    from src.data.porto import PortoPreprocessor, PortoDataset
    from sklearn.model_selection import train_test_split

    rng = np.random.RandomState(42)
    N = 500
    df = pd.DataFrame({
        'id': range(N),
        'target': rng.randint(0, 2, N),
        'ps_reg_01': rng.randn(N).astype(np.float32),
        'ps_reg_02': rng.randn(N).astype(np.float32),
        'ps_car_01_cat': rng.choice([-1, 1, 2, 3], N),
        'ps_ind_06_bin': rng.choice([-1, 0, 1], N),
    })
    df.loc[rng.choice(N, 20, replace=False), 'ps_reg_01'] = -1

    train_df, valid_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

    prep = PortoPreprocessor()
    train_ds = PortoDataset(prep.fit_transform(train_df))
    valid_ds = PortoDataset(prep.transform(valid_df))
    print(f'Synthetic data: train={len(train_ds)} valid={len(valid_ds)}')
    print(f'Numeric={prep.num_numeric}  Cat={prep.num_cat}  Bin={prep.num_bin}')

## 2. Tokenizer Behaviour Check

In [ ]:
from src.models.numeric_tokenizers import (
    LinearNumericTokenizer,
    CenterSoftBinTokenizer,
    OrderedThresholdTokenizer,
)

M = prep.num_numeric
D = 16
K = 8

torch.manual_seed(0)
x_demo = torch.randn(4, M)
mask_demo = torch.zeros(4, M, dtype=torch.bool)
mask_demo[0, 0] = True  # mark one entry as missing

for name, tok in [
    ('Linear',  LinearNumericTokenizer(M, D)),
    ('Center',  CenterSoftBinTokenizer(M, D, K)),
    ('Ordered', OrderedThresholdTokenizer(M, D, K)),
]:
    tokens, aux = tok(x_demo, mask_demo)
    print(f'{name:8s}  tokens.shape={tuple(tokens.shape)}  aux keys={list(aux.keys())}')

In [ ]:
# Verify ordered thresholds are monotone
ot = OrderedThresholdTokenizer(M, D, K)
thresholds = ot.get_thresholds().detach().numpy()  # (M, K)
diffs = np.diff(thresholds, axis=1)
print(f'Min inter-threshold gap: {diffs.min():.6f}  (should be > 0)')
assert (diffs > 0).all(), 'Thresholds are NOT monotone!'
print('All thresholds are strictly increasing ✓')

## 3. Training and Evaluation — Comparison

In [ ]:
from src.models.mixed_tabular_model import MixedTabularModel
from src.training.metrics import compute_metrics
from src.training.train_eval import set_seed, train_one_epoch, evaluate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 5        # keep short for notebook; increase for real runs
BATCH_SIZE = 512
LR = 1e-3

def build_model(mode: str) -> MixedTabularModel:
    return MixedTabularModel(
        num_numeric=prep.num_numeric,
        cat_vocab_sizes=prep.cat_vocab_sizes(),
        bin_vocab_sizes=prep.bin_vocab_sizes(),
        d_token=32,
        mlp_hidden=[128, 64],
        mode=mode,
        num_thresholds=8,
        num_centers=8,
        dropout=0.1,
    ).to(DEVICE)

y_valid_np = valid_ds.tensors['y'].numpy()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)

all_results = {}

for mode in ['linear', 'center', 'ordered']:
    set_seed(42)
    model = build_model(mode)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    history = []

    print(f'\n--- mode={mode} ---')
    for epoch in range(1, EPOCHS + 1):
        loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
        preds = evaluate(model, valid_loader, DEVICE)
        m = compute_metrics(y_valid_np, preds)
        history.append({'epoch': epoch, **m})
        print(f'  epoch {epoch}/{EPOCHS}  loss={loss:.4f}  AUC={m["auc"]:.4f}  LogLoss={m["logloss"]:.4f}')

    all_results[mode] = {'model': model, 'history': history, 'final': history[-1]}

In [ ]:
# Plot AUC comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = {'linear': 'steelblue', 'center': 'darkorange', 'ordered': 'forestgreen'}

for mode, res in all_results.items():
    epochs = [h['epoch'] for h in res['history']]
    aucs   = [h['auc'] for h in res['history']]
    losses = [h['logloss'] for h in res['history']]
    axes[0].plot(epochs, aucs, marker='o', label=mode, color=colors[mode])
    axes[1].plot(epochs, losses, marker='o', label=mode, color=colors[mode])

axes[0].set_title('Validation AUC'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Validation LogLoss'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Ordered Thresholds — Internal State Visualisation

In [ ]:
from src.training.train_eval import get_aux

ordered_model = all_results['ordered']['model']

# Learned thresholds
thresholds = ordered_model.numeric_tokenizer.get_thresholds().detach().cpu().numpy()  # (M, K)
print(f'Learned thresholds shape: {thresholds.shape}  (num_features × num_thresholds)')
print(f'Sample (feature 0): {thresholds[0].round(3)}')

In [ ]:
# Soft assignment visualisation for a subset of features
aux_data = get_aux(ordered_model, valid_loader, DEVICE)
assign_all = np.concatenate(aux_data['assign'], axis=0)  # (N, M, K)
assign_mean = assign_all.mean(axis=0)                     # (M, K)

n_show = min(8, prep.num_numeric)
fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2.5, 3), sharey=True)
if n_show == 1:
    axes = [axes]

for i in range(n_show):
    ax = axes[i]
    ax.bar(range(len(assign_mean[i])), assign_mean[i], color='forestgreen', alpha=0.8)
    ax.set_title(prep.numeric_cols[i] if i < len(prep.numeric_cols) else f'feat {i}', fontsize=8)
    ax.set_xlabel('Threshold index')

axes[0].set_ylabel('Mean assignment')
plt.suptitle('Mean soft assignment per threshold (ordered tokenizer)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Threshold heatmap (features × thresholds)
n_feat_show = min(20, prep.num_numeric)
fig, ax = plt.subplots(figsize=(10, max(3, n_feat_show * 0.4)))
im = ax.imshow(thresholds[:n_feat_show], aspect='auto', cmap='RdYlGn')
ax.set_xlabel('Threshold index')
ax.set_ylabel('Feature index')
ax.set_yticks(range(n_feat_show))
feat_labels = prep.numeric_cols[:n_feat_show] if n_feat_show <= len(prep.numeric_cols) else [f'feat {i}' for i in range(n_feat_show)]
ax.set_yticklabels(feat_labels, fontsize=7)
plt.colorbar(im, ax=ax, label='threshold value')
ax.set_title('Learned thresholds heatmap (first 20 numeric features)')
plt.tight_layout()
plt.show()

## 5. Result Summary

In [ ]:
import pandas as pd

rows = []
for mode, res in all_results.items():
    fin = res['final']
    rows.append({
        'mode': mode,
        'AUC': round(fin['auc'], 4),
        'LogLoss': round(fin['logloss'], 4),
        'RMSE': round(fin['rmse'], 4),
        'MAE': round(fin['mae'], 4),
        'R²': round(fin['r2'], 4),
    })

summary_df = pd.DataFrame(rows).set_index('mode')
print('=== Final Validation Metrics ===')
print(summary_df.to_string())

In [ ]:
# Bar chart summary
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

modes = list(all_results.keys())
aucs = [all_results[m]['final']['auc'] for m in modes]
lls  = [all_results[m]['final']['logloss'] for m in modes]

bar_colors = [colors[m] for m in modes]

axes[0].bar(modes, aucs, color=bar_colors)
axes[0].set_title('Final AUC (higher = better)')
axes[0].set_ylim(max(0, min(aucs) - 0.01), max(aucs) + 0.01)

axes[1].bar(modes, lls, color=bar_colors)
axes[1].set_title('Final LogLoss (lower = better)')
axes[1].set_ylim(max(0, min(lls) - 0.005), max(lls) + 0.005)

plt.tight_layout()
plt.show()

print('\nDone.')